# Utility Operations Intelligence — Configuration

**Single source of truth** for the whole demo. The orchestrator and every step notebook load this with `%run ./config` (or `%run ../config`), so changing a value here reconfigures the entire pipeline. This is the **only** place you need to edit before running.

## Parameters

Edit the values below, then re-run this cell (and the Setup cells in the orchestrator). Every step reads them as plain Python variables via `%run`.

- **Steps 1-4** need `CATALOG`, `SCHEMA`, `VOLUME_NAME`, `WAREHOUSE_ID`, `USER_EMAIL`.
- **`GENIE_SPACE_IDS`** is filled in automatically by Step 3 (and printed so you can paste it here for reproducible re-runs).
- **`SUPERVISOR_ENDPOINT` / `KA_ENDPOINTS`** come from the Step 5 UI build — paste them here (or into the Step 5 capture cell) before running Step 6.

In [0]:
# =====================================================================================
# Demo configuration — the SINGLE place to set values for this demo.
# =====================================================================================

# --- your environment: EDIT THESE ---
CATALOG            = "serverless_stable_umpuol_catalog"   # Unity Catalog for all demo data
SCHEMA             = "kailyn_klaassen"                    # schema inside that catalog
VOLUME_NAME        = "reports"                            # UC Volume for the 50 PDFs
WAREHOUSE_ID       = "d6a9e269c6d0868b"                   # SQL warehouse (Genie spaces run on it)
USER_EMAIL         = "kailyn.klaassen@databricks.com"     # Genie space parent path + app owner
DATABRICKS_PROFILE = "fe-vm-fevm-serverless-stable-umpuol"  # used for LOCAL runs only

# --- agent resources: filled in as you complete Steps 3 and 5 ---
# Step 3 sets these automatically; paste them back here to make re-runs reproducible.
GENIE_SPACE_IDS = {
    "Grid Operations":        "01f17fc43b901e3999607bb7ff313695",
    "Financial Performance":  "01f17fc43c0a1a70bb10f470e43164c1",
    "Maintenance & Workforce":"01f17fc43c7e17d698dc3d9f2ee49adb",
}

# Step 5 (UI) produces these serving-endpoint names — paste them in:
SUPERVISOR_ENDPOINT = "mas-fffa4146-endpoint"   # paste the mas-*-endpoint from Step 5b
KA_ENDPOINTS        = ['ka-8a460225-endpoint']   # paste the ka-*-endpoint(s) from Step 5a

# --- sensible defaults: usually fine as-is ---
APP_NAME = "utility-ops-supervisor"


## Step registry & roadmap

In [0]:
# Demo steps, in order. Steps 5a/5b are UI-only (Agent Bricks); everything else is scripted.
STEPS = [
    {"id": 1,  "name": "Generate source data (13 tables, ~11.5M rows)", "notebook": "1-data/generate"},
    {"id": "1b","name": "Sanity SQL checks",                            "notebook": "1-data/sanity_sql"},
    {"id": 2,  "name": "Build rollups + 4 metric views",               "notebook": "2-metric-views/build_metric_views"},
    {"id": "2b","name": "Validate 55 measures",                         "notebook": "2-metric-views/test_metric_views"},
    {"id": 3,  "name": "Create 3 Genie spaces",                         "notebook": "3-genie-spaces/build_genie_spaces"},
    {"id": 4,  "name": "Upload 50 PDFs to UC Volume",                   "notebook": "4-documents/anchor_narrative"},
    {"id": "5a","name": "Build Knowledge Assistant (UI)",               "notebook": None},
    {"id": "5b","name": "Build Supervisor Multi-Agent System (UI)",     "notebook": None},
    {"id": 6,  "name": "Grant app SP + deploy app",                     "notebook": "6-app/deploy_app"},
    {"id": 7,  "name": "The narrative & example questions to ask the app","notebook": None},
]


## Shared helpers

Spark, the workspace client, schema/volume setup, and the **grant-then-deploy** helpers used by Step 6.

In [0]:
def get_spark():
    """Ambient Spark in Databricks; serverless Databricks Connect when run locally."""
    from pyspark.sql import SparkSession
    active = SparkSession.getActiveSession()
    if active is not None:
        return active
    from databricks.connect import DatabricksSession
    return DatabricksSession.builder.profile(DATABRICKS_PROFILE).serverless().getOrCreate()


def workspace_client():
    """Ambient WorkspaceClient in Databricks; profile-based when run locally."""
    from databricks.sdk import WorkspaceClient
    from pyspark.sql import SparkSession
    if SparkSession.getActiveSession() is not None:
        return WorkspaceClient()
    return WorkspaceClient(profile=DATABRICKS_PROFILE)


def ensure_schema_and_volume():
    """Create the target schema + PDF volume if they do not exist."""
    spark = get_spark()
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}')
    print(f'schema {CATALOG}.{SCHEMA} and volume /{VOLUME_NAME} ready')


In [0]:
import os

# Local filesystem root of this demo (the folder containing 1-data/, 6-app/, config.ipynb).
# Used only for LOCAL runs; in Databricks the notebook path is auto-detected.
DEMO_ROOT = "/Users/kailyn.klaassen/utility-ops-intelligence-demo"


def demo_root():
    """Return the demo root folder. In Databricks, derive it from the running notebook path;
    locally, use DEMO_ROOT."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        p = ctx.notebookPath().get()
        # config lives at the demo root, so its parent dir is the root
        return '/Workspace' + os.path.dirname(p)
    except Exception:
        return DEMO_ROOT


def app_sp_client_id(app_name=None):
    """The Databricks App's service principal client id (its runtime identity)."""
    w = workspace_client()
    return w.apps.get(app_name or APP_NAME).service_principal_client_id


def _patch(path, body):
    return workspace_client().api_client.do('PATCH', path, body=body)


def _endpoint_id(name):
    return workspace_client().serving_endpoints.get(name).id


def grant_app_sp(app_name=None):
    """Grant the app's service principal everything the supervisor reaches THROUGH it:
    CAN_QUERY on supervisor + KA endpoints, CAN_RUN on the Genie spaces, CAN_USE on the
    warehouse, and UC data access on catalog+schema. Idempotent. Run BEFORE deploying so
    the app comes up already authorized (no runtime 403s)."""
    sp = app_sp_client_id(app_name)
    print(f'App service principal: {sp}')

    # 1. Serving endpoints -> CAN_QUERY
    endpoints = [SUPERVISOR_ENDPOINT] + list(KA_ENDPOINTS or [])
    for name in dict.fromkeys([e for e in endpoints if e]):
        try:
            eid = _endpoint_id(name)
            _patch(f'/api/2.0/permissions/serving-endpoints/{eid}',
                   {'access_control_list': [{'service_principal_name': sp, 'permission_level': 'CAN_QUERY'}]})
            print(f'  OK   CAN_QUERY endpoint {name}')
        except Exception as e:
            print(f'  FAIL endpoint {name}: {e}')

    # 2. Genie spaces -> CAN_RUN
    for label, sid in (GENIE_SPACE_IDS or {}).items():
        try:
            _patch(f'/api/2.0/permissions/genie/{sid}',
                   {'access_control_list': [{'service_principal_name': sp, 'permission_level': 'CAN_RUN'}]})
            print(f'  OK   CAN_RUN genie {label} ({sid})')
        except Exception as e:
            print(f'  FAIL genie {label}: {e}')

    # 3. SQL warehouse -> CAN_USE
    try:
        _patch(f'/api/2.0/permissions/warehouses/{WAREHOUSE_ID}',
               {'access_control_list': [{'service_principal_name': sp, 'permission_level': 'CAN_USE'}]})
        print(f'  OK   CAN_USE warehouse {WAREHOUSE_ID}')
    except Exception as e:
        print(f'  FAIL warehouse: {e}')

    # 4. Unity Catalog data access (SELECT on schema cascades to all tables + metric views)
    try:
        _patch(f'/api/2.1/unity-catalog/permissions/catalog/{CATALOG}',
               {'changes': [{'principal': sp, 'add': ['USE_CATALOG','USE_SCHEMA','SELECT','READ_VOLUME','EXECUTE']}]})
        _patch(f'/api/2.1/unity-catalog/permissions/schema/{CATALOG}.{SCHEMA}',
               {'changes': [{'principal': sp, 'add': ['USE_SCHEMA','SELECT','READ_VOLUME','EXECUTE']}]})
        print(f'  OK   UC grants on {CATALOG} and {CATALOG}.{SCHEMA}')
    except Exception as e:
        print(f'  FAIL UC grants: {e}')

    print('Grants complete.')


In [0]:
def app_url(app_name=None):
    try:
        return workspace_client().apps.get(app_name or APP_NAME).url
    except Exception:
        return '(deploy the app first)'


def describe():
    print('Utility Operations Intelligence — configuration')
    print(f'  Catalog/Schema : {CATALOG}.{SCHEMA}')
    print(f'  Volume         : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}')
    print(f'  Warehouse      : {WAREHOUSE_ID}')
    print(f'  Genie spaces   : {list(GENIE_SPACE_IDS.values())}')
    print(f'  Supervisor     : {SUPERVISOR_ENDPOINT}')
    print(f'  KA endpoints   : {KA_ENDPOINTS}')
    print(f'  App            : {APP_NAME}')


In [0]:
describe()